In [ ]:
# CELL 1: Install Libraries (Sửa lỗi FSDP)

# 1. Gỡ cài đặt các bản có thể bị xung đột
!pip uninstall -y torch torchvision torchaudio transformers accelerate

# 2. Cài đặt PyTorch stack (torch, torchaudio) cho CUDA 11.8
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118

# 3. Cài đặt torchvision TƯƠNG THÍCH (rất quan trọng)
!pip install -q torchvision --index-url https://download.pytorch.org/whl/cu118

# 4. Cài đặt các thư viện khác SAU KHI PyTorch đã ổn định
# Cài đặt transformers và accelerate phiên bản MỚI NHẤT
!pip install -q transformers
!pip install -q accelerate

# Cài các thư viện còn lại
!pip install -q pytorch-crf
!pip install -q scikit-learn tqdm numpy pandas

print("\n--- Cài đặt hoàn tất! Vui lòng KHỞI ĐỘNG LẠI KERNEL và chạy lại các cell. ---")

In [1]:
# CELL 2: Load Data (Dataset Mới - SemEval)
import json

# SỬA ĐỔI: Đặt đường dẫn chính xác tới 3 file .json mới của bạn
TRAIN_FILE = '/kaggle/input/mams-dataset/train.json'
TEST_FILE = '/kaggle/input/mams-dataset/test.json'
VALID_FILE = '/kaggle/input/mams-dataset/valid.json'

def load_data(fname):
    with open(fname, 'r', encoding='utf-8') as f:
        return json.load(f)

# Nạp dữ liệu mới
train_data_json = load_data(TRAIN_FILE)
val_data_json = load_data(VALID_FILE)
test_data_json = load_data(TEST_FILE)

print(f"Loaded {len(train_data_json)} training samples.")
print(f"Loaded {len(val_data_json)} validation samples.")
print(f"Loaded {len(test_data_json)} test samples.")

Loaded 4297 training samples.
Loaded 500 validation samples.
Loaded 500 test samples.


In [4]:
# CELL MỚI: BÁO CÁO CHI TIẾT DỮ LIỆU (THÊM SAU CELL 2)
import pandas as pd
from IPython.display import display, HTML
import matplotlib.pyplot as plt

print("="*80)
print("BÁO CÁO CHI TIẾT: CẤU TRÚC DỮ LIỆU ABSA – MAMS DATASET")
print("="*80)
print(f"Dataset: MAMS (SemEval) | Train: 4297 | Val: 500 | Test: 500")
print(f"Nguồn: Kaggle – Multi-Aspect Multi-Sentiment Dataset")
print("-"*80)

# === LẤY 5 MẪU ĐẠI DIỆN ===
samples = train_data_json[:5]
colors = {'positive': '#4CAF50', 'neutral': '#FFC107', 'negative': '#F44336'}

for idx, sample in enumerate(samples):
    print(f"\nMẪU {idx+1}:")
    print(f"   Câu gốc: {' '.join(sample['token'])}")
    print(f"   Số từ: {len(sample['token'])} | Số aspect: {len(sample['aspects'])}")
    
    # Bảng aspect
    if sample['aspects']:
        df = pd.DataFrame(sample['aspects'])
        df['aspect_term'] = [' '.join(sample['token'][row['from']:row['to']]) for _, row in df.iterrows()]
        df = df[['aspect_term', 'from', 'to', 'polarity']]
        df.columns = ['Aspect Term', 'From', 'To', 'Polarity']
        display(df.style.set_table_styles([{'selector': 'th', 'props': [('font-weight', 'bold')]}]))
    else:
        print("   → Không có aspect nào được đánh dấu.")
    
    # Trực quan hóa trong câu
    sentence = sample['token'].copy()
    html = "<p style='font-family: monospace; font-size: 15px; line-height: 2.2;'>"
    for i, word in enumerate(sentence):
        colored = False
        for asp in sample['aspects']:
            if asp['from'] <= i < asp['to']:
                color = colors.get(asp['polarity'], '#9E9E9E')
                html += f"<span style='background-color:{color}; padding:2px 6px; border-radius:4px; color:white; font-weight:bold;'>{word}</span> "
                colored = True
                break
        if not colored:
            html += f"{word} "
    html += "</p>"
    
    # Chú thích
    legend = "<div style='margin:8px 0; font-size:13px;'>"
    for pol, col in colors.items():
        legend += f"<span style='background-color:{col}; padding:2px 8px; border-radius:3px; color:white; margin-right:12px;'>{pol.upper()}</span>"
    legend += "</div>"
    
    display(HTML(html + legend))
    print("-"*80)

BÁO CÁO CHI TIẾT: CẤU TRÚC DỮ LIỆU ABSA – MAMS DATASET
Dataset: MAMS (SemEval) | Train: 4297 | Val: 500 | Test: 500
Nguồn: Kaggle – Multi-Aspect Multi-Sentiment Dataset
--------------------------------------------------------------------------------

MẪU 1:
   Câu gốc: the decor is not special at all but their food and amazing prices make up for it .
   Số từ: 18 | Số aspect: 3


,Aspect Term,From,To,Polarity
0,decor,1,2,negative
1,food,9,10,positive
2,prices,12,13,positive


--------------------------------------------------------------------------------

MẪU 2:
   Câu gốc: when tables opened up , the manager sat another party before us .
   Số từ: 13 | Số aspect: 2


,Aspect Term,From,To,Polarity
0,tables,1,2,neutral
1,manager,6,7,negative


--------------------------------------------------------------------------------

MẪU 3:
   Câu gốc: though the menu includes some unorthodox offerings ( a peanut butter roll , for instance ) , the classics are pure and great -- we 've never had better sushi anywhere , including japan .
   Số từ: 35 | Số aspect: 4


...

--------------------------------------------------------------------------------

MẪU 4:
   Câu gốc: service is good although a bit in your face , we were asked every five mins if food was ok , but better that than being ignored .
   Số từ: 28 | Số aspect: 2


,Aspect Term,From,To,Polarity
0,service,0,1,positive
1,food,17,18,neutral


--------------------------------------------------------------------------------

MẪU 5:
   Câu gốc: ps- i just went for brunch on saturday and the eggs served with onions and rosemary were amazing .
   Số từ: 19 | Số aspect: 2


,Aspect Term,From,To,Polarity
0,brunch,5,6,neutral
1,eggs served with onions,10,14,positive


--------------------------------------------------------------------------------


In [6]:
# Thêm vào cuối cell để phân tích thống kê
aspect_count = [len(s['aspects']) for s in train_data_json]
polarity_count = {'positive': 0, 'neutral': 0, 'negative': 0}
for s in train_data_json:
    for a in s['aspects']:
        polarity_count[a['polarity']] += 1

print(f"\nTHỐNG KÊ TOÀN BỘ TRAIN SET (n = {len(train_data_json)}):")
print(f"   • Trung bình aspect: {sum(aspect_count)/len(aspect_count):.2f}")
print(f"   • Max aspect: {max(aspect_count)}")
print(f"   • Phân bố polarity:")
for p, c in polarity_count.items():
    print(f"     – {p.upper():8}: {c} ({c/sum(polarity_count.values())*100:.1f}%)")


THỐNG KÊ TOÀN BỘ TRAIN SET (n = 4297):
   • Trung bình aspect: 2.60
   • Max aspect: 11
   • Phân bố polarity:
     – POSITIVE: 3380 (30.2%)
     – NEUTRAL : 5042 (45.1%)
     – NEGATIVE: 2764 (24.7%)


In [2]:
# CELL 3: (VIẾT LẠI) BERT Preprocessing (Span-Based ASC)
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm
from transformers import BertTokenizer

# --- 1. Định nghĩa Nhãn và Model BERT ---
BERT_MODEL_NAME = 'bert-base-uncased'

# ATE tags
tag2id = {'O': 0, 'B-ASP': 1, 'I-ASP': 2}
id2tag = {v: k for k, v in tag2id.items()}

# ASC tags
sentiment2id = {'positive': 0, 'neutral': 1, 'negative': 2}
id2sent = {v: k for k, v in sentiment2id.items()}

IGNORE_INDEX = -100
max_len = 100 # Max len cho BERT
MAX_ASPECTS = 11 # Số lượng aspect tối đa trong 1 câu (để padding)

# --- 2. Khởi tạo Tokenizer ---
tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)
vocab_size = tokenizer.vocab_size
print(f"BERT Vocab size: {vocab_size}")

# --- 3. Hàm Preprocessing Mới ---
def create_bert_samples(json_data, max_seq_len):
    all_samples = []
    
    for sample in tqdm(json_data, desc="Preprocessing"):
        original_tokens = sample['token']
        
        # 1. Tokenize (WordPiece) và Align nhãn ATE (B-I-O)
        subword_input_ids = []
        subword_ate_tags = []
        # word_map[i] = j nghĩa là subword thứ i thuộc về word gốc thứ j
        word_map = [] 
        
        for word_idx, word in enumerate(original_tokens):
            subwords = tokenizer.tokenize(word)
            if not subwords: subwords = [tokenizer.unk_token]
            
            subword_input_ids.extend(subwords)
            word_map.extend([word_idx] * len(subwords))

        # Cắt bớt nếu quá dài (trước khi thêm [CLS], [SEP])
        if len(subword_input_ids) > max_seq_len - 2:
            subword_input_ids = subword_input_ids[:max_seq_len - 2]
            word_map = word_map[:max_seq_len - 2]

        # Thêm [CLS], [SEP]
        final_input_tokens = [tokenizer.cls_token] + subword_input_ids + [tokenizer.sep_token]
        # word_map: -1 cho [CLS] và [SEP]
        final_word_map = [-1] + word_map + [-1] 
        
        # Chuyển token -> IDs
        numeric_input_ids = tokenizer.convert_tokens_to_ids(final_input_tokens)
        
        # Attention Mask
        attention_mask = [1] * len(numeric_input_ids)
        
        # 2. Xử lý Nhãn ATE (token-level) và Nhãn ASC (span-level)
        ate_tags = [tag2id['O']] * max_seq_len
        
        span_indices = [[0, 0]] * MAX_ASPECTS # (start, end)
        span_sentiments = [IGNORE_INDEX] * MAX_ASPECTS
        span_mask = [0] * MAX_ASPECTS

        aspect_count = 0
        for asp in sample['aspects']:
            if aspect_count >= MAX_ASPECTS:
                break # Bỏ qua nếu quá nhiều aspect
                
            polarity = asp['polarity']
            if polarity not in sentiment2id:
                continue

            # Chỉ số (word-level)
            b_word_idx = asp['from']
            e_word_idx = asp['to'] - 1 

            # Tìm chỉ số (subword-level) tương ứng
            try:
                # +1 vì có [CLS] ở đầu
                b_sub_idx = final_word_map.index(b_word_idx) 
                # Tìm vị trí cuối cùng của end_word_idx
                e_sub_idx = len(final_word_map) - 1 - final_word_map[::-1].index(e_word_idx)
            except ValueError:
                continue # Bỏ qua aspect nếu không tìm thấy (ví dụ: do bị cắt)

            # Gán nhãn ATE (B-I-O)
            ate_tags[b_sub_idx] = tag2id['B-ASP']
            for i in range(b_sub_idx + 1, e_sub_idx + 1):
                ate_tags[i] = tag2id['I-ASP']
            
            # Gán nhãn ASC (span-level)
            span_indices[aspect_count] = [b_sub_idx, e_sub_idx]
            span_sentiments[aspect_count] = sentiment2id[polarity]
            span_mask[aspect_count] = 1
            aspect_count += 1

        # 3. Padding
        pad_len = max_seq_len - len(numeric_input_ids)
        numeric_input_ids += [tokenizer.pad_token_id] * pad_len
        attention_mask += [0] * pad_len
        
        all_samples.append({
            'input_ids': torch.tensor(numeric_input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.bool),
            'ate_tags': torch.tensor(ate_tags, dtype=torch.long),           # Nhãn ATE
            'span_indices': torch.tensor(span_indices, dtype=torch.long),   # (Max_Asp, 2)
            'span_sentiments': torch.tensor(span_sentiments, dtype=torch.long), # (Max_Asp)
            'span_mask': torch.tensor(span_mask, dtype=torch.bool)          # (Max_Asp)
        })

    return all_samples

# Process
train_data = create_bert_samples(train_data_json, max_len)
val_data = create_bert_samples(val_data_json, max_len)
test_data = create_bert_samples(test_data_json, max_len)

# --- 4. Tạo Dataloader (Giữ nguyên) ---
train_loader = DataLoader(train_data, batch_size=16, shuffle=True) # Giảm batch size xuống 16
val_loader = DataLoader(val_data, batch_size=16)
test_loader = DataLoader(test_data, batch_size=16)

print(f"\nTotal samples: Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")
print("Sample check:", train_data[0]['input_ids'].shape, train_data[0]['ate_tags'].shape, train_data[0]['span_indices'].shape)

BERT Vocab size: 30522


Preprocessing: 100%|██████████| 500/500 [00:00<00:00, 899.96it/s]


Total samples: Train: 4297, Val: 500, Test: 500
Sample check: torch.Size([100]) torch.Size([100]) torch.Size([10, 2])


In [3]:
# CELL 4: (VIẾT LẠI) Model BERT (Span Representation Nâng cao)
import torch
import torch.nn as nn
from torchcrf import CRF
from transformers import BertModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class JointBERT_Span_ABSA(nn.Module):
    def __init__(self, bert_model_name, num_ate_tags, num_asc_classes, dropout=0.1):
        super().__init__()
        self.num_ate_tags = num_ate_tags
        self.num_asc_classes = num_asc_classes
        
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.bert_dropout = nn.Dropout(dropout)
        bert_hidden_size = self.bert.config.hidden_size # 768
        
        # 1. ATE Head (Token-level)
        self.ate_projection = nn.Linear(bert_hidden_size, num_ate_tags)
        self.crf = CRF(num_ate_tags, batch_first=True)
        
        # 2. ASC Head (Span-level)
        # SỬA ĐỔI: Input là (start_token + end_token + mean_pool)
        self.asc_projection = nn.Linear(bert_hidden_size * 3, num_asc_classes)
        self.asc_dropout = nn.Dropout(dropout) # Thêm Dropout cho ASC head

    def _get_span_representation(self, hidden_state, span_indices, span_mask):
        """
        Trích xuất vector đại diện cho span.
        hidden_state: (batch_size, seq_len, 768)
        span_indices: (batch_size, MAX_ASPECTS, 2)
        span_mask: (batch_size, MAX_ASPECTS)
        """
        batch_size, max_aspects, _ = span_indices.shape
        bert_hidden_size = hidden_state.shape[-1]
        
        # (batch_size, MAX_ASPECTS, bert_hidden_size * 3)
        span_representations = torch.zeros(batch_size, max_aspects, bert_hidden_size * 3).to(device)
        
        for i in range(batch_size):
            sample_hidden_state = hidden_state[i] # (seq_len, 768)
            sample_spans = span_indices[i]
            sample_span_mask = span_mask[i]
            
            valid_span_count = 0
            for j in range(max_aspects):
                if sample_span_mask[j] == 0:
                    continue # Span này là padding
                        
                start, end = sample_spans[j]
                
                # 1. Lấy vector BẮT ĐẦU (start)
                start_rep = sample_hidden_state[start]
                
                # 2. Lấy vector KẾT THÚC (end)
                end_rep = sample_hidden_state[end]
                
                # 3. Lấy vector TRUNG BÌNH (mean)
                if start == end:
                    mean_rep = start_rep
                else:
                    aspect_vectors = sample_hidden_state[start : end + 1]
                    mean_rep = torch.mean(aspect_vectors, dim=0)
                
                # Ghép 3 vector lại
                combined_rep = torch.cat([start_rep, end_rep, mean_rep])
                span_representations[i, valid_span_count] = combined_rep
                valid_span_count += 1
                
        return span_representations

    def forward(self, input_ids, attention_mask, span_indices=None, span_mask=None):
        bert_outputs = self.bert(input_ids, attention_mask=attention_mask)
        last_hidden_state = bert_outputs.last_hidden_state
        last_hidden_state = self.bert_dropout(last_hidden_state)
        
        # --- 1. ATE Path ---
        emissions = self.ate_projection(last_hidden_state)
        
        # --- 2. ASC Path ---
        sent_logits = None
        if span_indices is not None:
            # SỬA ĐỔI: Dùng hàm trích xuất mới
            span_reps = self._get_span_representation(last_hidden_state, span_indices, span_mask)
            span_reps = self.asc_dropout(span_reps)
            sent_logits = self.asc_projection(span_reps)
            
        return sent_logits, emissions

    def decode_tags(self, emissions, mask):
        return self.crf.decode(emissions, mask=mask)

    def loss_fn(self, sent_logits, emissions, ate_tags, span_sentiments, span_mask, mask, asc_weight=0.5, ate_weight=0.5):
        # 1. ATE Loss (CRF Loss)
        ate_loss = -self.crf(emissions, ate_tags, mask=mask, reduction='mean')
        
        # 2. ASC Loss (CrossEntropyLoss)
        sent_loss_fn = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
        active_logits = sent_logits.view(-1, self.num_asc_classes)
        active_labels = span_sentiments.view(-1)
        asc_loss = sent_loss_fn(active_logits, active_labels)
        
        # SỬA ĐỔI: Thử cân bằng lại loss 50/50
        return (asc_weight * asc_loss) + (ate_weight * ate_loss)

# Khởi tạo model
model = JointBERT_Span_ABSA(
    bert_model_name=BERT_MODEL_NAME,
    num_ate_tags=len(tag2id),
    num_asc_classes=len(sentiment2id)
).to(device)

print(f"BERT Model (Advanced Span-Rep) loaded on {device}")

2025-11-09 03:31:48.415300: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762659108.597601     149 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762659108.644901     149 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Using device: cuda


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BERT Model (Advanced Span-Rep) loaded on cuda


In [ ]:
# CELL 5: (VIẾT LẠI) BERT Training (Hyperparameter Mới)
import torch
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score

# --- 1. Hyperparameters (Sửa đổi) ---
num_epochs = 10 
bert_lr = 2e-5

# SỬA ĐỔI: Giảm LR của các lớp mới xuống để chống Overfitting
other_lr = 2e-4 # <--- Giảm từ 1e-3 (gấp 5 lần)

no_decay = ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {'params': [p for n, p in model.named_parameters() if 'bert' in n and not any(nd in n for nd in no_decay)], 
     'weight_decay': 0.01, 'lr': bert_lr},
    {'params': [p for n, p in model.named_parameters() if 'bert' in n and any(nd in n for nd in no_decay)], 
     'weight_decay': 0.0, 'lr': bert_lr},
    {'params': [p for n, p in model.named_parameters() if 'bert' not in n], 
     'weight_decay': 0.01, 'lr': other_lr} # <--- Áp dụng other_lr mới
]
optimizer = AdamW(optimizer_grouped_parameters)

total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, 
                                            num_warmup_steps=total_steps * 0.1, 
                                            num_training_steps=total_steps)

best_avg_f1 = 0.0

# --- 2. Vòng lặp Training (Sửa đổi) ---
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} Train"):
        optimizer.zero_grad()
        
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        ate_tags = batch['ate_tags'].to(device)
        span_indices = batch['span_indices'].to(device)
        span_sentiments = batch['span_sentiments'].to(device)
        span_mask = batch['span_mask'].to(device)
        
        sent_logits, emissions = model(ids, mask, span_indices, span_mask)
        
        # SỬA ĐỔI: Thử cân bằng lại loss 50/50
        loss = model.loss_fn(sent_logits, emissions, ate_tags, span_sentiments, span_mask, mask,
                             asc_weight=0.5, 
                             ate_weight=0.5)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    
    # --- 3. Validation (Giữ nguyên) ---
    model.eval()
    ate_preds_val, ate_true_val = [], []
    asc_preds_val, asc_true_val = [], []
    val_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} Val"):
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            ate_tags = batch['ate_tags'].to(device)
            span_indices = batch['span_indices'].to(device)
            span_sentiments = batch['span_sentiments'].to(device)
            span_mask = batch['span_mask'].to(device)
            
            sent_logits, emissions = model(ids, mask, span_indices, span_mask)
            
            loss = model.loss_fn(sent_logits, emissions, ate_tags, span_sentiments, span_mask, mask,
                                 asc_weight=0.5, ate_weight=0.5)
            val_loss += loss.item()
            
            decoded_tags_batch = model.decode_tags(emissions, mask=mask)
            asc_preds_batch = torch.argmax(sent_logits, dim=2)
            
            for i in range(len(ids)):
                seq_len = mask[i].sum().item()
                
                ate_pred_seq = decoded_tags_batch[i][:seq_len]
                ate_true_seq = ate_tags[i][:seq_len].cpu().numpy()
                ate_preds_val.extend(ate_pred_seq)
                ate_true_val.extend(ate_true_seq)

                sample_span_preds = asc_preds_batch[i].cpu().numpy()
                sample_span_true = span_sentiments[i].cpu().numpy()
                sample_span_mask = span_mask[i].cpu().numpy()
                
                for j in range(MAX_ASPECTS):
                    if sample_span_mask[j]:
                        asc_preds_val.append(sample_span_preds[j])
                        asc_true_val.append(sample_span_true[j])

    # --- 4. Báo cáo Kết quả (Giữ nguyên) ---
    avg_train_loss = total_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    ate_f1_val = f1_score(ate_true_val, ate_preds_val, average='macro', zero_division=0)
    
    if len(asc_true_val) == 0:
        asc_f1_val, asc_acc_val = 0.0, 0.0
    else:
        asc_acc_val = accuracy_score(asc_true_val, asc_preds_val)
        asc_f1_val = f1_score(asc_true_val, asc_preds_val, average='macro', zero_division=0)
    
    avg_f1 = (ate_f1_val + asc_f1_val) / 2

    print(f"\nEpoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"  [Val] Aspect F1 (ATE): {ate_f1_val:.4f}")
    print(f"  [Val] Sentiment Acc: {asc_acc_val:.4f} | Sentiment F1: {asc_f1_val:.4f}")
    
    if avg_f1 > best_avg_f1:
        best_avg_f1 = avg_f1
        torch.save(model.state_dict(), '/kaggle/working/best_absa_bert_span_v2.pth')
        print(f"  → Best model saved! (Avg F1: {avg_f1:.4f})")

In [4]:
# CELL 6: (VIẾT LẠI) Final Evaluation on Test Set
from sklearn.metrics import classification_report

print("Loading best model from /kaggle/working/best_absa_bert_span_v2.pth")
model.load_state_dict(torch.load('/kaggle/working/best_absa_bert_span_v2.pth'))
model.eval()

ate_preds_test, ate_true_test = [], []
asc_preds_test, asc_true_test = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Running on Test Set"):
        # Nạp data từ batch
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        ate_tags = batch['ate_tags'].to(device)
        span_indices = batch['span_indices'].to(device)
        span_sentiments = batch['span_sentiments'].to(device)
        span_mask = batch['span_mask'].to(device)
        
        # Chạy model
        sent_logits, emissions = model(ids, mask, span_indices, span_mask)
        
        # Lấy ATE predictions (Token-level)
        decoded_tags_batch = model.decode_tags(emissions, mask=mask)
        
        # Lấy ASC predictions (Span-level)
        asc_preds_batch = torch.argmax(sent_logits, dim=2)
        
        for i in range(len(ids)):
            seq_len = mask[i].sum().item()
            
            # ATE F1 (Token-level, bỏ padding)
            ate_pred_seq = decoded_tags_batch[i][:seq_len]
            ate_true_seq = ate_tags[i][:seq_len].cpu().numpy()
            ate_preds_test.extend(ate_pred_seq)
            ate_true_test.extend(ate_true_seq)

            # ASC F1 (Span-level, dùng span_mask)
            sample_span_preds = asc_preds_batch[i].cpu().numpy()
            sample_span_true = span_sentiments[i].cpu().numpy()
            sample_span_mask = span_mask[i].cpu().numpy()
            
            for j in range(MAX_ASPECTS):
                if sample_span_mask[j]: # Chỉ check span thật
                    asc_preds_test.append(sample_span_preds[j])
                    asc_true_test.append(sample_span_true[j])

print("\n" + "="*60)
print(" FINAL RESULTS (TEST SET) ")
print("="*60)

# --- ATE Results ---
print("\n--- Aspect Term Extraction (ATE) Results ---")
target_ate_ids = sorted(list(set(ate_true_test)))
target_ate_names = [id2tag.get(i, 'UNK') for i in target_ate_ids]
print(classification_report(ate_true_test, ate_preds_test, target_names=target_ate_names, digits=4, zero_division=0))

# --- ASC Results ---
print("\n--- Aspect Sentiment Classification (ASC) Results ---")
target_asc_ids = sorted(list(set(asc_true_test)))
target_asc_names = [id2sent.get(i, 'UNK') for i in target_asc_ids]
print(classification_report(asc_true_test, asc_preds_test, target_names=target_asc_names, digits=4, zero_division=0))

print("="*60)

Loading best model from /kaggle/working/best_absa_bert_span_v2.pth


Running on Test Set: 100%|██████████| 32/32 [00:03<00:00, 10.02it/s]


 FINAL RESULTS (TEST SET) 

--- Aspect Term Extraction (ATE) Results ---
              precision    recall  f1-score   support

           O     0.9706    0.9591    0.9648     13970
       B-ASP     0.7638    0.8285    0.7948      1335
       I-ASP     0.6601    0.7031    0.6809       815

    accuracy                         0.9353     16120
   macro avg     0.7982    0.8302    0.8135     16120
weighted avg     0.9378    0.9353    0.9364     16120


--- Aspect Sentiment Classification (ASC) Results ---
              precision    recall  f1-score   support

    positive     0.8015    0.7794    0.7903       399
     neutral     0.8509    0.8369    0.8439       607
    negative     0.7714    0.8207    0.7953       329

    accuracy                         0.8157      1335
   macro avg     0.8080    0.8123    0.8098      1335
weighted avg     0.8166    0.8157    0.8159      1335



In [ ]:
# CELL 7: (VIẾT LẠI) Inference (Demo cho model Span-Based)

def predict_bert_span(text):
    model.eval()
    
    # 1. Tokenize (BERT Tokenizer)
    inputs = tokenizer(text, 
                       return_tensors="pt", 
                       max_length=max_len, 
                       padding="max_length", 
                       truncation=True)
    
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device).bool() # Chuyển sang bool cho CRF
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    
    # === PASS 1: Chạy ATE (Lấy Aspect Spans) ===
    with torch.no_grad():
        # Không truyền span_indices, model sẽ chỉ chạy ATE
        sent_logits_pass1, emissions = model(input_ids, attention_mask, span_indices=None)
    
    # sent_logits_pass1 sẽ là None
    
    # Decode ATE (B-I-O)
    ate_tags_decoded = model.decode_tags(emissions, mask=attention_mask)[0]
    
    # 2. Parse Spans (Tìm [start, end] từ B-I-O)
    found_spans = [] # List các [start_idx, end_idx]
    current_span = []
    
    for i in range(1, len(tokens)): # Bỏ [CLS]
        token = tokens[i]
        ate_tag_id = ate_tags_decoded[i]
        
        if token == tokenizer.sep_token or token == tokenizer.pad_token:
            break
            
        ate_tag_str = id2tag[ate_tag_id]
        
        if ate_tag_str == 'B-ASP':
            if current_span: # Đóng span cũ
                found_spans.append([current_span[0], current_span[-1]])
            current_span = [i] # Mở span mới
        elif ate_tag_str == 'I-ASP':
            if current_span:
                current_span.append(i)
        else: # 'O'
            if current_span: # Đóng span
                found_spans.append([current_span[0], current_span[-1]])
            current_span = []
    
    if current_span: # Đóng span cuối cùng
        found_spans.append([current_span[0], current_span[-1]])

    if not found_spans:
        return "No aspects found."
        
    # 3. Chuẩn bị cho Pass 2
    num_found = len(found_spans)
    
    # Pad các span tìm được
    padded_spans = found_spans + [[0, 0]] * (MAX_ASPECTS - num_found)
    padded_spans = padded_spans[:MAX_ASPECTS] # Cắt nếu > MAX_ASPECTS
    
    # Tạo tensor
    span_indices_tensor = torch.tensor([padded_spans], dtype=torch.long).to(device)
    span_mask_tensor = torch.tensor([[1] * num_found + [0] * (MAX_ASPECTS - num_found)], dtype=torch.bool).to(device)
    span_mask_tensor = span_mask_tensor[:, :MAX_ASPECTS] # Cắt
    
    # === PASS 2: Chạy ASC (Lấy Sentiment) ===
    with torch.no_grad():
        # Chạy model LẦN NỮA, lần này truyền span
        sent_logits, _ = model(input_ids, attention_mask, 
                               span_indices=span_indices_tensor, 
                               span_mask=span_mask_tensor)
    
    # Lấy dự đoán
    # (1, MAX_ASPECTS, num_classes) -> (1, MAX_ASPECTS)
    asc_preds = torch.argmax(sent_logits, dim=2)[0] # Lấy batch 0
    
    # 4. Reconstruct (Ghép sub-word và sentiment)
    results = []
    for i in range(num_found):
        start, end = found_spans[i]
        
        # Ghép sub-word lại
        aspect_tokens = tokens[start : end + 1]
        term = tokenizer.convert_tokens_to_string(aspect_tokens)
        
        # Lấy sentiment
        polarity_id = asc_preds[i].item()
        polarity_str = id2sent[polarity_id]
        
        results.append((term, polarity_str))
            
    return results

# Demo
test_sentences = [
    "The food was amazing but the service was terrible.",
    "Decent price, awful taste.",
    "The ambiance is not special at all but their food and amazing prices make up for it.",
    "I love the food but hate the service.",
    "The staff is friendly and the decor is lovely."
]

print("="*70)
print("BERT (Span-Based v2) - END-TO-END ABSA DEMO")
print("="*70)

# Nạp model tốt nhất
model.load_state_dict(torch.load('/kaggle/working/best_absa_bert_span_v2.pth'))
for s in test_sentences:
    aspects = predict_bert_span(s)
    print(f"Text: {s}")
    print(f"   → Aspects: {aspects}\n")